<a href="https://colab.research.google.com/github/marudhu099/Hands-On-Large-Language-Models/blob/main/Generating_first_text.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This cell installs the `transformers` and `accelerate` libraries, which are essential for working with Hugging Face models.

In [ ]:
!pip install -U transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 113.4 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.13.1
    Uninstalling transformers-5.13.1:
      Successfully uninstalled transformers-5.13.1


This cell imports the necessary libraries, detects the available device (GPU or CPU), and then loads the `Phi-3-mini-4k-instruct` model and its tokenizer from Hugging Face.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, AutoModel

model_id = "microsoft/Phi-3-mini-4k-instruct"

# Detect if a GPU is available, otherwise use CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Use native implementation for stability
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map=device,
    torch_dtype="auto",
    trust_remote_code=False
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
print(f"Setup complete! Model and tokenizer loaded.")

Using device: cuda


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

Setup complete! Model and tokenizer loaded.


This cell demonstrates text generation using the loaded model. It sets up a `pipeline` for text generation, defines specific generation arguments, and then uses the model to generate a joke based on a user message.

In [ ]:
messages = [
    {"role": "user", "content": "Create a funny joke about chickens."}
]

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

generation_args = {
    "max_new_tokens": 500,
    "return_full_text": False,
    "temperature": 0.7,
    "do_sample": True,
    "clean_up_tokenization_spaces": True
}

# Generate output
output = pipe(messages, **generation_args)
print(output[0]['generated_text'])

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Why did the chicken join a band? Because it had the drumsticks!


This cell showcases another text generation example. It prepares a prompt, tokenizes it, and then uses the model's `generate` method to produce a short response, which is then decoded and printed.

In [45]:
prompt = "Write an email apologizing to Sarah for the tragic gardening mishap.Explain how it happened.<|assistant|>"

device = "cuda" if torch.cuda.is_available() else "cpu"

input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

# for id in input_ids[0]:
#  print(tokenizer.decode(id))

generation_output = model.generate(
 input_ids=input_ids,
 max_new_tokens=50
)
# Print the output
print(tokenizer.decode(generation_output[0]))
# print(tokenizer.decode([3323, 622]), " <---- Specific Token words")

Write an email apologizing to Sarah for the tragic gardening mishap.Explain how it happened.<|assistant|> Subject: Sincere Apologies for the Gardening Mishap


Dear Sarah,


I hope this message finds you well. I am writing to express my deepest apologies for the unfortunate incident that


This cell defines a utility function `show_tokens` to visualize how a given sentence is broken down into tokens by the tokenizer. It uses different colors to highlight individual tokens and then calls this function with a sample text and the loaded tokenizer.

In [42]:
colors_list = [
 '102;194;165', '252;141;98', '141;160;203',
 '231;138;195', '166;216;84', '255;217;47'
]

text = """
English and CAPITALIZATION
🎵鸟
show_tokens False None elif == >= else: two tabs:" " Three tabs: " "
12.0*50=600
"""

def show_tokens(sentence, tokenizer_name):
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
    token_ids = tokenizer(sentence).input_ids

    # 1. Print the raw token IDs list first
    print(f"Token IDs: {token_ids}\n")

    # 2. Then the colored tokens, each with its ID underneath-style
    for idx, t in enumerate(token_ids):
        print(
            f'\x1b[0;30;48;2;{colors_list[idx % len(colors_list)]}m' +
            tokenizer.decode(t) + f':{t}' +
            '\x1b[0m',
            end=' '
        )

show_tokens(text, "microsoft/Phi-3-mini-4k-instruct")
# show_tokens(text, "Xenova/gpt-4")

Token IDs: [29871, 13, 24636, 322, 315, 3301, 1806, 1964, 26664, 8098, 13, 243, 162, 145, 184, 236, 187, 162, 13, 4294, 29918, 517, 12360, 7700, 6213, 25342, 1275, 6736, 1683, 29901, 1023, 18859, 6160, 376, 12753, 18859, 29901, 376, 376, 13, 29896, 29906, 29889, 29900, 29930, 29945, 29900, 29922, 29953, 29900, 29900, 13]

:29871 
:13 English:24636 and:322 C:315 AP:3301 IT:1806 AL:1964 IZ:26664 ATION:8098 
:13 �:243 �:162 �:145 �:184 �:236 �:187 �:162 
:13 show:4294 _:29918 to:517 kens:12360 False:7700 None:6213 elif:25342 ==:1275 >=:6736 else:1683 ::29901 two:1023 tabs:18859 :":6160 ":376 Three:12753 tabs:18859 ::29901 ":376 ":376 
:13 1:29896 2:29906 .:29889 0:29900 *:29930 5:29945 0:29900 =:29922 6:29953 0:29900 0:29900 
:13 

In [41]:
tok = AutoTokenizer.from_pretrained("Xenova/gpt-4")

# find the emoji's token IDs
ids = tok(" 🎵").input_ids
print(ids)              # the pieces

# now rebuild the emoji from the pieces!
print(tok.decode(ids))  # 🎵 comes back!

[11410, 236, 113]
 🎵


**TEXT EMBEDDINGS**

In [7]:
from transformers import AutoModel, AutoTokenizer

# This cell demonstrates how to load a pre-trained tokenizer and model from Hugging Face Transformers,
# tokenize a simple sentence ('Hello world'), and then process these tokens through the model
# to obtain their corresponding text embeddings. Finally, it prints the shape of the resulting embeddings.

tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-base")
model = AutoModel.from_pretrained("microsoft/deberta-v3-xsmall")

tokens = tokenizer('Hello world', return_tensors='pt')

# print token ids
# print(tokens.input_ids)
# print(tokenizer.decode(tokens.input_ids[0]))
output = model(**tokens)[0]

print(output.shape)

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-xsmall
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
mask_predictions.dense.bias                | UNEXPECTED |  | 
mask_predictions.classifier.bias           | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight          | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED |  | 
mask_predictions.dense.weight              | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias            | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED |  | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED |  | 
lm_predictions.lm_head.bias                | UNEXPECTED |  | 
mask_predictions.classifier.weight         | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from d

tensor([[    1, 31414,   232,     2]])
[CLS]Hello world[SEP]
torch.Size([1, 4, 384])


In [2]:
# time to go text embeddings sentence_transformer
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

sentences = [
    "The weather is lovely today.",
    "It's so sunny outside!",
    "He drove to the stadium.",
]
embeddings = model.encode(sentences)
print(embeddings.shape, "embeddings shape")
print(embeddings, "embeddings")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

(3, 384) embeddings shape
[[ 0.01919577  0.12008539  0.15959834 ... -0.00536285 -0.08109502
   0.05021336]
 [-0.01869039  0.0415187   0.07431544 ...  0.00486599 -0.06190438
   0.03187511]
 [ 0.136502    0.08227322 -0.02526163 ...  0.08762044  0.03045843
  -0.01075751]] embeddings


In [8]:
# word embedding with using library of Gensim top modelling for humans

!pip install gensim
import gensim.downloader as api

model = api.load("glove-wiki-gigaword-50")

# Correct way to perform the analogy 'king - man + women'
model.most_similar(positive=['king', 'women'], negative=['man'])

[('kingdom', 0.701315701007843),
 ('queen', 0.6152784824371338),
 ('invited', 0.6111606359481812),
 ('ii', 0.5973405241966248),
 ('vii', 0.5971130132675171),
 ('ceremonies', 0.5901867747306824),
 ('commonwealth', 0.5839658975601196),
 ('viii', 0.5788934826850891),
 ('throne', 0.5778504014015198),
 ('attend', 0.5759660005569458)]

Let's break down the code in the following cell step by step:

*   **`import pandas as pd`**: Imports the pandas library, commonly used for data manipulation and analysis, and assigns it the alias `pd` for convenience.
*   **`from urllib import request`**: Imports the `request` module from the `urllib` library, which is used for opening and reading URLs (web pages, files from a URL).
*   **`data = request.urlopen('https://storage.googleapis.com/maps-premium/dataset/yes_complete/train.txt')`**: This line opens a URL pointing to a text file named `train.txt` on a Google Cloud Storage bucket. The content of this file (which presumably contains raw playlist data) is assigned to the `data` variable.
*   **`lines = data.read().decode("utf-8").split('\n')[2:]`**: This processes the raw data:
    *   `data.read()`: Reads the entire content from the opened URL.
    *   `.decode("utf-8")`: Decodes the raw bytes into a UTF-8 string.
    *   `.split('\n')`: Splits the entire string into a list of individual lines, using the newline character `\n` as the delimiter.
    *   `[2:]`: Slices the list to skip the first two lines, implying these lines contain metadata or headers that are not part of the actual playlist data.
*   **`playlists = [s.rstrip().split() for s in lines if len(s.split()) > 1]`**: This line processes the `lines` to create the `playlists` list:
    *   `for s in lines`: Iterates through each `s` (line) in the `lines` list.
    *   `s.rstrip()`: Removes any trailing whitespace (like newline characters) from the current line.
    *   `s.rstrip().split()`: Splits the cleaned line into a list of strings, using whitespace as the delimiter. This assumes that song IDs within a playlist are separated by spaces.
    *   `if len(s.split()) > 1`: This is a filter that includes only those lines (playlists) that contain more than one song ID. Playlists with a single song are discarded.
*   **`print("one", playlists[0])`** and **`print("two", playlists[1])`**: These lines print the first two processed playlists to show their structure.
*   **`songs_file = request.urlopen('https://storage.googleapis.com/maps-premium/dataset/yes_complete/song_hash.txt')`**: Similar to the `train.txt` file, this opens another URL, this time pointing to `song_hash.txt`, which likely contains metadata about the songs.
*   **`songs_file = songs_file.read().decode("utf-8").split('\n')`**: Reads and decodes the content of `song_hash.txt` and splits it into individual lines.
*   **`songs = [s.rstrip().split('\t') for s in songs_file]`**: Processes `songs_file` to create a list of song details:
    *   `for s in songs_file`: Iterates through each line in the `songs_file`.
    *   `s.rstrip()`: Removes trailing whitespace.
    *   `s.rstrip().split('\t')`: Splits each line into a list of strings, using the tab character `\t` as the delimiter. This suggests that song details (like ID, title, artist) are tab-separated.
*   **`songs_df = pd.DataFrame(data=songs, columns = ['id', 'title', 'artist'])`**: Creates a pandas DataFrame named `songs_df` from the `songs` list. It assigns column names 'id', 'title', and 'artist'.
*   **`songs_df = songs_df.set_index('id')`**: Sets the 'id' column as the DataFrame's index. This allows for quick lookup of song details by their ID.
*   **`print( 'Playlist #1:\n ', playlists[0], '\n')`** and **`print( 'Playlist #2:\n ', playlists[1])`**: These lines print the first two playlists again for verification after all processing steps, ensuring the data looks as expected.

In [15]:
# Training song model

import pandas as pd
from urllib import request

data = request.urlopen('https://storage.googleapis.com/maps-premium/dataset/yes_complete/train.txt')

# Parse the playlist dataset file. Skip the first two lines as
# they only contain metadata
lines = data.read().decode("utf-8").split('\n')[2:]

# Remove playlists with only one song
playlists = [s.rstrip().split() for s in lines if len(s.split()) > 1]
print("one", playlists[0])
print("two", playlists[1])

# Load song metadata
songs_file = request.urlopen('https://storage.googleapis.com/maps-premium/dataset/yes_complete/song_hash.txt')
songs_file = songs_file.read().decode("utf-8").split('\n')
songs = [s.rstrip().split('\t') for s in songs_file]
songs_df = pd.DataFrame(data=songs, columns = ['id', 'title', 'artist'])
songs_df = songs_df.set_index('id')

print( 'Playlist #1:\n ', playlists[0], '\n')
print( 'Playlist #2:\n ', playlists[1])

one ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '2', '42', '43', '44', '45', '46', '47', '48', '20', '49', '8', '50', '51', '52', '53', '54', '55', '56', '57', '25', '58', '59', '60', '61', '62', '3', '63', '64', '65', '66', '46', '47', '67', '2', '48', '68', '69', '70', '57', '50', '71', '72', '53', '73', '25', '74', '59', '20', '46', '75', '76', '77', '59', '20', '43']
two ['78', '79', '80', '3', '62', '81', '14', '82', '48', '83', '84', '17', '85', '86', '87', '88', '74', '89', '90', '91', '4', '73', '62', '92', '17', '53', '59', '93', '94', '51', '50', '27', '95', '48', '96', '97', '98', '99', '100', '57', '101', '102', '25', '103', '3', '104', '105', '106', '107', '47', '108', '109', '110', '111', '112', '113', '25', '63', '62', '114', '115', '84', '116', '117', '118', '119', '120', '1

\The cell below trains a `Word2Vec` model using the `gensim` library. `Word2Vec` is a popular technique for learning word embeddings, which are dense vector representations of words that capture their semantic meaning based on their context.

Here's a breakdown of the `Word2Vec` parameters used:

-   **`playlists`**: This is the corpus on which the model is trained. In this case, it's a list of playlists, where each playlist is a list of song IDs. `Word2Vec` treats each playlist as a 'sentence' and each song ID as a 'word'. The model learns relationships between song IDs that frequently appear together in playlists.
-   **`vector_size=32`**: This parameter defines the dimensionality of the word vectors (embeddings). Each song ID will be represented as a 32-dimensional vector. A larger `vector_size` can capture more nuanced relationships but requires more training data and computational resources.
-   **`window=20`**: This is the maximum distance between the current and predicted word within a sentence. In simple terms, it defines the size of the context window. For each song ID, the model considers up to 20 song IDs before and after it in a playlist as its context. A larger window allows the model to capture broader contextual relationships.
-   **`negative=50`**: This parameter specifies the number of 'negative' (randomly sampled) words that will be used for negative sampling. Negative sampling is an optimization technique where, instead of updating all weights for all non-context words, only a small number of negative words are updated. A higher number typically improves the quality of the embeddings but also increases training time.
-   **`min_count=1`**: This parameter ignores all words with a total frequency lower than this value. Here, `min_count=1` means that all song IDs, even those appearing only once in the entire dataset, will be included in the vocabulary and used for training. If a song ID appears less than `min_count` times, it will be ignored.
-   **`workers=4`**: This sets the number of worker threads to use for training the model. Utilizing multiple workers can speed up the training process by parallelizing computations.

In [13]:
# Let's train the model

from gensim.models import Word2Vec

# Fix: Renamed 'odel' to 'song_w2v_model' for clarity and to avoid conflict
song_w2v_model = Word2Vec(playlists, vector_size=32, window=20, negative=50, min_count=1, workers=4)

song_id = 2172
# Ask the model for songs similar to song #2172
# Fix: Using the correct variable for the newly trained model and accessing its .wv attribute
song_w2v_model.wv.most_similar(positive=str(song_id))

# Print song
print(songs_df.iloc[2172])

title     Fade To Black
artist        Metallica
Name: 2172 , dtype: object


In [21]:
import numpy as np

def print_recommendations(song_id):
 similar_songs = np.array(song_w2v_model.wv.most_similar(positive=str(song_id),topn=5))[:,0]
 return songs_df.iloc[similar_songs]
# Extract recommendations

print_recommendations(3)

,title,artist
id,,
62,Runaway (w\/ Pusha T),Kanye West
59,"Monster (w\/ Rick Ross, Jay-Z, Nicki Minaj & B...",Kanye West
486,On The The Next One (feat. Swizz Beatz),Jay-Z
81,Fancy (w\/ T.I. & Swizz Beatz),Drake
19613,Deja Vu (w\/ Jay-Z),Beyonce
